In [1]:
import pandas as pd
import plotly.express as px
from datetime import datetime
from collections import Counter
import re

## Acronyms & nomenclature
* `Rec`: Réception (ou enregistrement).
* `Inv`: Investigation (ou examen).

## Sample size N = 110

## 1. Read initial file sent to me

In [2]:
df = pd.read_excel('Everything merged 18.12._new.xlsx')
print(df.shape)
df.head()

(110, 71)


,PatientID,DateRecInvIRM,ResultatIRMRecInv,PIRADSLesionRecInv,ResultatBiopsieRecInv,VolProstateIRM,PSA,Column8,2nd mri date,2nd mri resultat,...,all patients.age_capra,all patients.D28VolD90,all patients.D28VolV100,pet.PatientID,pet.DateRecInvRadiative,pet.TypeRecInvRadiative,pet.ResultatRecInvRadiative,date treatment.PatientId,date treatment.TypeTX,date treatment.DateBrachy
0,5753253,2024-05-18,Positive,4.0,Positive,27.7,4.10,NaN,NaT,NaN,...,1,140.9,89.3,5753253.0,2024-09-12,PET ... type,Positif,5753253,Curietherapie LDR,2022-02-24
1,5725736,2024-01-13,Négative,NaN,NaN,NaN,6.00,NaN,NaT,NaN,...,1,141.3,89.6,NaN,NaT,NaN,NaN,5725736,Curietherapie LDR,2022-01-04
2,808919,2024-02-04,Négative,NaN,NaN,30.4,5.87,NaN,NaT,NaN,...,1,164.2,95.4,NaN,NaT,NaN,NaN,808919,Curietherapie LDR,2021-05-27
3,5687019,2023-11-18,Négative,3.0,Négative,18.0,4.20,NaN,NaT,NaN,...,1,157.0,94.2,NaN,NaT,NaN,NaN,5687019,Curietherapie LDR,2021-03-25
4,5339331,2023-06-24,Positive,NaN,NaN,28.0,3.78,NaN,NaT,NaN,...,1,159.0,96.0,5339331.0,2023-08-03,PET ... type,Positif,5339331,Curietherapie LDR,2021-03-11


## 2. Remove empty columns
9 empty columns removed: `['Column8', 'Column12', 'Column15', 'Column18', 'Column19', '4th mri date', 'Column25', 'RecInvbiopsie.PSA', 'all patients.causedeces']`

In [3]:
empty_cols = df.columns[df.isna().all()].tolist()
print(empty_cols)

['Column8', 'Column12', 'Column15', 'Column18', 'Column19', '4th mri date', 'Column25', 'RecInvbiopsie.PSA', 'all patients.causedeces']


In [4]:
df = df.drop(columns=empty_cols)

## 3. Remove duplicate columns
5 duplicate columns removed.

In [5]:
cols = df.columns
remove_cols = set()

for i in range(len(cols)):
    if cols[i] in remove_cols:
        continue

    col1 = df[cols[i]]

    for j in range(i + 1, len(cols)):
        if cols[j] in remove_cols:
            continue

        col2 = df[cols[j]]

        # Only compare overlapping non-NaN values
        mask = col1.notna() & col2.notna()

        # Require at least one overlapping value
        if mask.sum() > 0 and (col1[mask] == col2[mask]).all():
            print(f"{col1.name} SAME AS {col2.name}")
            remove_cols.add(cols[j])

PatientID SAME AS RecInvbiopsie.PatientID
PatientID SAME AS all patients.PatientId
PatientID SAME AS pet.PatientID
PatientID SAME AS date treatment.PatientId
ResultatIRMRecInv SAME AS ResultatBiopsieRecInv
3rdmri vol SAME AS 4th mri vol
4th mri psa  SAME AS RecInvbiopsie.GleasonSecRecInv
RecInvbiopsie.GleasonPrimRecInv SAME AS RecInvbiopsie.Column10
all patients.TypeTX SAME AS date treatment.TypeTX


In [6]:
remove_cols = ['RecInvbiopsie.PatientID', 'all patients.PatientId', 'pet.PatientID', 
               'date treatment.PatientId', 'date treatment.TypeTX']

In [7]:
df = df.drop(columns=remove_cols)

In [8]:
df.shape

(110, 57)

## 4. Rename & Re-Order Columns

_More automated way to rename the columns (but I am not using this currently)._
```
col_list = list(df.columns)
df = df.rename(columns=lambda x: x.replace("all patients.", ""))
col_list = list(df.columns)
col_list = [re.sub(r'([a-z])([A-Z])', r'\1_\2', col).lower() for col in col_list]
col_list = [s.rstrip() for s in col_list]
col_list = [s.replace("  ", " ") for s in col_list]
col_list = [s.replace(" ", "_") for s in col_list]
col_list = [s.replace("aps", "psa") for s in col_list]
col_list = [s.replace("treatment", "tx") for s in col_list]
col_list = [s.replace("biopsie", "biopsy") for s in col_list]
col_list = [s.replace("resultat", "result") for s in col_list]

# Rename dataframe columns
df.columns = col_list
```

In [9]:
col_map = {
    "PatientID": "patient_id",
    "all patients.Age": "tx-age",
    "all patients.ÉchantPélevés": "tx-biopsy_num_samples",
    "all patients.ÉchantPositifs": "tx-biopsy_num_positive",
    "all patients.GleasonTotal": "tx-gleason_total",
    "all patients.T": "tx-t_stage",
    "all patients.protocole": "tx-protocol",
    "date treatment.DateBrachy": "tx-date",
    "all patients.TypeTX": "tx-type",
    "all patients.DoseTotalProstate": "tx-total_dose_prostate",
    "all patients.D28VolD90": "tx-d28_vol_d90",
    "all patients.D28VolV100": "tx-d28_vol_v100",
    "all patients.ADT": "tx-adt",
    "all patients.ApsMonth": "psa-time_since_tx",
    "all patients.Aps": "psa-val",
    "all patients.nadiraps02": "psa-nadir_02",
    "all patients.nadiraps05": "psa-nadir_05",
    "all patients.CAPRA": "psa-capra_total",
    "all patients.aps_capra": "psa-capra_psa",
    "all patients.gleason_capra": "psa-capra_gleason",
    "all patients.tstage_capra": "psa-capra_t_stage",
    "all patients.biopsy_capra": "psa-capra_biopsy",
    "all patients.age_capra": "psa-capra_age",
    "all patients.ddeces": "date_death",
    "all patients.biochemical recurrence": "bf-date",

    "DateRecInvIRM": "mri_1-date",
    "ResultatIRMRecInv": "mri_1-result",
    "PIRADSLesionRecInv": "mri_1-pirads_score",
    "VolProstateIRM": "mri_1-prostate_vol",
    "PSA": "mri_1-psa",

    "2nd mri date": "mri_2-date",
    "2nd mri resultat": "mri_2-result",
    "2ndmri  vol": "mri_2-prostate_vol",
    "2nd mri psa": "mri_2-psa",
    
    "3rdmri date": "mri_3-date",
    "3rdmri resultat": "mri_3-result",
    "3rdmri vol": "mri_3-prostate_vol",
    "3rdmri psa ": "mri_3-psa",

    "4th mri date ": "mri_4-date",
    "4thmri resultat": "mri_4-result",
    "4th mri vol": "mri_4-prostate_vol",
    "4th mri psa ": "mri_4-psa",

    "RecInvbiopsie.DateRecInvBiopsie": "biopsy-date",
    "RecInvbiopsie.ResultatRecInvBiopsie": "biopsy-result",
    "RecInvbiopsie.GleasonPrimRecInv": "biopsy-gleason_1ary",
    "RecInvbiopsie.GleasonSecRecInv": "biopsy-gleason_2ary",
    "RecInvbiopsie.GleasonTertRecInv": "biopsy-gleason_3ary",

    "pet.DateRecInvRadiative": "pet-date",
    "pet.TypeRecInvRadiative":"pet-tracer",
    "pet.ResultatRecInvRadiative": "pet-result",   
}

In [10]:
df = df.rename(columns=col_map)

In [11]:
df.to_csv("df-clean-01.csv", index=False)
df.to_excel("df-clean-01.xlsx", index=False)